# 模型 2：kNN

**负责人：** 2 号　**阶段：** 3 模型

句向量 → kNN（余弦距离）。

## 任务

标"（选做）"的任务在必做部分完成后再做。

- [x] 输入先做 L2 归一化；k 取 3–51；比较 uniform 和 distance 两种加权
- [x] FastText 输入要用对应折的版本：第 k 折用 `_fold{k}`，预测测试集用 `_full`
- [x] 标签噪声检查：找出近邻多数标签和自己不一样的样本
- [x] 每次实验先用 `new_run_id()` 生成 run_id，评分记录和预测文件都带上同一个 run_id

## 产出

- `results/m2_knn.csv`
- 用 `save_preds(run_id, ...)` 保存 OOF 概率和测试集概率（存到 `DATA_ROOT/preds/`）

In [ ]:
import sys
from pathlib import Path

# 向上查找仓库根目录（含 src/config.py 的文件夹），在任何子文件夹里打开都能导入 src
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("找不到仓库根目录：请在 clone 下来的仓库里启动 Jupyter（Colab 先 %cd 进仓库）")
sys.path.insert(0, str(ROOT))

from src.config import DATA_ROOT, FIGURES_DIR, LABELS, SEED                 # 路径和常量
from src.data import load_clean, load_raw, load_splits                      # 读取数据和划分
from src.evaluate import evaluate, new_run_id, plot_confusion, save_result  # 统一评分
from src.cache import load_embedding, save_embedding, save_preds            # 嵌入和预测结果的读写

print("仓库根目录:", ROOT)
print("DATA_ROOT:", DATA_ROOT)

## Approach

kNN is the instance-based model in our set: a tweet gets the majority label of its *k* most similar training tweets, using cosine similarity between sentence vectors. It has no training step, so it shows how much the embedding alone separates the classes.

- inputs: FastText mean and SIF vectors (fold-wise, from `11_emb_fasttext`) and Sentence-BERT vectors (from `10_emb_sbert`, once member 1 delivers them)
- tuning: *k* from 3 to 51 and uniform vs distance weighting, chosen by 5-fold cross-validated macro-F1
- each embedding's best setting is then refitted on the whole training split and scored once on the test split

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.neighbors import KNeighborsClassifier

from src.config import CLEAN_DIR, CLEAN_VERSION, EMB_DIR, N_FOLDS

clean = load_clean()
splits = load_splits(clean=clean)
label_of = dict(zip(clean["id"], clean["label"]))
dev = splits[splits["split"] == "train"].reset_index(drop=True)
test = splits[splits["split"] == "test"].reset_index(drop=True)
y_dev = dev["id"].map(label_of).to_numpy()
y_test = test["id"].map(label_of).to_numpy()

# embedding name -> True if it was trained per fold (self-trained), False if one frozen copy serves every fold
EMBEDDINGS = {"fasttext_mean": True, "fasttext_sif": True, "sbert_multi": False, "sbert_en": False}
available = {}
for name, foldwise in EMBEDDINGS.items():
    probe = EMB_DIR / (f"{name}_fold0.npy" if foldwise else f"{name}.npy")
    if probe.exists():
        available[name] = foldwise
    else:
        print(f"not available yet: {name} ({probe.name})")
print("using:", list(available))

In [ ]:
def fold_matrices(name, foldwise, k):
    """Training-part and validation-part vectors for fold k, L2-normalised."""
    source = f"{name}_fold{k}" if foldwise else name
    train_ids = dev.loc[dev["fold"] != k, "id"]
    val_ids = dev.loc[dev["fold"] == k, "id"]
    return normalise(load_embedding(source, ids=train_ids)), normalise(load_embedding(source, ids=val_ids))


def full_matrices(name, foldwise):
    """Whole training split and test split, from the version trained on all training tweets."""
    source = f"{name}_full" if foldwise else name
    return normalise(load_embedding(source, ids=dev["id"])), normalise(load_embedding(source, ids=test["id"]))


def normalise(x):
    x = np.asarray(x, dtype=np.float32)
    return x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-12, None)


def oof_proba(name, foldwise, params, keep_by_fold=None):
    """Out-of-fold class probabilities for every training tweet.

    keep_by_fold[k] (optional) is a boolean mask over the training split; rows that are False are left out of
    fold k's training part. Validation rows are always scored.
    """
    proba = np.zeros((len(dev), 3))
    for k in range(N_FOLDS):
        in_val = (dev["fold"] == k).to_numpy()
        keep = ~in_val if keep_by_fold is None else (~in_val & keep_by_fold[k])
        x_train, x_val = fold_matrices(name, foldwise, k)
        x_train = x_train[keep[~in_val]]
        clf = KNeighborsClassifier(metric="cosine", algorithm="brute", **params).fit(x_train, y_dev[keep])
        proba[in_val] = clf.predict_proba(x_val)
    return proba

## Tuning *k* and the weighting

In [ ]:
K_VALUES = list(range(3, 52, 2))
rows = []
for name, foldwise in available.items():
    for weights in ("uniform", "distance"):
        for k in K_VALUES:
            pred = oof_proba(name, foldwise, {"n_neighbors": k, "weights": weights}).argmax(1)
            rows.append({"embedding": name, "weights": weights, "k": k, "macro_f1": f1_score(y_dev, pred, average="macro")})
tuning = pd.DataFrame(rows)
best = tuning.loc[tuning.groupby("embedding")["macro_f1"].idxmax()].reset_index(drop=True)
best

In [ ]:
SURFACE, INK, INK_2, AXIS, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#c3c2b7", "#e1e0d9"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # categorical slots 1-4, fixed order

fig, ax = plt.subplots(figsize=(7, 3.2), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
for color, (name, row) in zip(SERIES, best.set_index("embedding").iterrows()):
    curve = tuning[(tuning["embedding"] == name) & (tuning["weights"] == row["weights"])]
    ax.plot(curve["k"], curve["macro_f1"], color=color, linewidth=2, label=f"{name} ({row['weights']})")
    ax.scatter([row["k"]], [row["macro_f1"]], color=color, s=40, zorder=3, edgecolor=SURFACE, linewidth=2)
    ax.annotate(f"{row['macro_f1']:.3f}", (row["k"], row["macro_f1"]), textcoords="offset points", xytext=(0, 8),
                ha="center", fontsize=8, color=INK_2)
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(AXIS)
ax.tick_params(colors=INK_2, labelsize=8)
ax.set_xlabel("k (neighbours)", color=INK_2, fontsize=9)
ax.set_ylabel("CV macro-F1", color=INK_2, fontsize=9)
ax.set_title("kNN: cross-validated macro-F1 by number of neighbours", loc="left", fontsize=11, color=INK)
ax.legend(frameon=False, fontsize=8, labelcolor=INK_2, loc="lower right")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "m2_knn_k_curve.png", dpi=200, facecolor=SURFACE)
plt.show()

## Final runs: cross-validation and test score for each embedding

Each embedding's best setting gets one run: its out-of-fold predictions are scored as `cv`, the model refitted on the whole training split is scored once on `test`, and both probability sets are saved under the same `run_id` for the ensemble.

In [ ]:
import time

final = []
for _, row in best.iterrows():
    name, foldwise = row["embedding"], available[row["embedding"]]
    params = {"n_neighbors": int(row["k"]), "weights": row["weights"]}
    run_id = new_run_id("knn", name)

    start = time.time()
    oof = oof_proba(name, foldwise, params)
    x_dev, x_test = full_matrices(name, foldwise)
    clf = KNeighborsClassifier(metric="cosine", algorithm="brute", **params).fit(x_dev, y_dev)
    test_proba = clf.predict_proba(x_test)
    seconds = time.time() - start

    for split, y_true, proba in (("cv", y_dev, oof), ("test", y_test, test_proba)):
        save_result(evaluate(y_true, proba.argmax(1), run_id=run_id, model="knn", embedding=name, split=split,
                             params=params, train_seconds=seconds), "m2_knn")
    save_preds(run_id, oof_ids=dev["id"], oof_proba=oof, test_ids=test["id"], test_proba=test_proba, classes=clf.classes_)
    final.append({"run_id": run_id, "embedding": name, **params,
                  "cv macro-F1": f1_score(y_dev, oof.argmax(1), average="macro"),
                  "test macro-F1": f1_score(y_test, test_proba.argmax(1), average="macro")})
final = pd.DataFrame(final).round(3)
final

In [ ]:
top = final.sort_values("cv macro-F1", ascending=False).iloc[0]
x_dev, x_test = full_matrices(top["embedding"], available[top["embedding"]])
clf = KNeighborsClassifier(metric="cosine", algorithm="brute", n_neighbors=int(top["n_neighbors"]),
                           weights=top["weights"]).fit(x_dev, y_dev)
fig = plot_confusion(y_test, clf.predict(x_test), f"kNN ({top['embedding']}) on the test split",
                     save_as=FIGURES_DIR / "m2_knn_confusion.png")

## Label-noise check with neighbours

A simple rule such as "most neighbours have a different label" flags far too many tweets when the embedding is weak: with three classes, a random neighbour already disagrees about two times in three. We therefore use a simplified **confident learning** rule (Northcutt et al., 2021) on held-out kNN probabilities:

- for each class *c*, the threshold *t_c* is the average predicted probability of *c* among tweets labelled *c*;
- a tweet labelled *i* is flagged when its probability for *i* is below *t_i* **and** the model confidently predicts another class *j* (probability ≥ *t_j*).

The probabilities come from an inner cross-validation **inside each outer fold's training part**, so the outer validation labels never decide what gets removed. We then compare the flags with the VADER disagreements from `31_label_noise_errors` and retrain kNN without the flagged tweets.

In [ ]:
name, foldwise = top["embedding"], available[top["embedding"]]
params = {"n_neighbors": int(top["n_neighbors"]), "weights": top["weights"]}


def confident_learning_flags(proba, y):
    """Flag rows whose given label looks wrong given held-out class probabilities."""
    thresholds = np.array([proba[y == c, c].mean() for c in range(proba.shape[1])])
    predicted = proba.argmax(1)
    confident_other = proba[np.arange(len(y)), predicted] >= thresholds[predicted]
    return (proba[np.arange(len(y)), y] < thresholds[y]) & (predicted != y) & confident_other


keep_by_fold = {}
suspicious = np.zeros(len(dev), dtype=bool)  # flagged in at least one outer fold, for reporting
for k in range(N_FOLDS):
    in_val = (dev["fold"] == k).to_numpy()
    x_train, _ = fold_matrices(name, foldwise, k)
    y_train, inner_folds = y_dev[~in_val], dev.loc[~in_val, "fold"].to_numpy()
    inner_proba = np.zeros((len(y_train), 3))
    for j in np.unique(inner_folds):
        held = inner_folds == j
        clf = KNeighborsClassifier(metric="cosine", algorithm="brute", **params).fit(x_train[~held], y_train[~held])
        inner_proba[held] = clf.predict_proba(x_train[held])
    flagged = confident_learning_flags(inner_proba, y_train)
    keep = np.ones(len(dev), dtype=bool)
    keep[np.flatnonzero(~in_val)[flagged]] = False
    keep_by_fold[k] = keep
    suspicious |= ~keep
per_fold = np.mean([(~keep_by_fold[k])[(dev["fold"] != k).to_numpy()].mean() for k in range(N_FOLDS)])
print(f"embedding: {name} | flagged per fold: {per_fold:.1%} of the training part on average | "
      f"flagged in at least one fold: {suspicious.sum()} of {len(dev)} ({suspicious.mean():.1%})")

vader_file = CLEAN_DIR / f"vader_disagreements_v{CLEAN_VERSION}.csv"
if vader_file.exists():
    vader_ids = set(pd.read_csv(vader_file)["id"])
    in_vader = dev["id"].isin(vader_ids).to_numpy()
    print(f"VADER disagreements in the training split: {in_vader.sum()}; "
          f"of these flagged by kNN: {(in_vader & suspicious).sum()} ({(in_vader & suspicious).sum() / max(in_vader.sum(), 1):.0%}); "
          f"flag rate among the other tweets: {suspicious[~in_vader].mean():.0%}")

In [ ]:
baseline_f1 = f1_score(y_dev, oof_proba(name, foldwise, params).argmax(1), average="macro")
filtered_pred = oof_proba(name, foldwise, params, keep_by_fold=keep_by_fold).argmax(1)
filtered_f1 = f1_score(y_dev, filtered_pred, average="macro")
save_result(evaluate(y_dev, filtered_pred, run_id=new_run_id("knn", name), model="knn", embedding=name, split="cv",
                     params={**params, "noise_filter": "confident_learning"},
                     notes="training folds without tweets flagged by confident learning"), "m2_knn")
pd.DataFrame({"training data": ["all tweets", "without flagged tweets"],
              "CV macro-F1": [baseline_f1, filtered_f1]}).round(3)

**How to read this.** Removing flagged tweets from the **training** folds only (validation tweets are always scored) tells us whether the questionable labels hurt the model. The validation folds still contain the given labels, so the score measures agreement with those labels, not with true sentiment.

Confident learning assumes the model is right most of the time. With a weak embedding (FastText kNN reaches a CV macro-F1 of about 0.5) the flags mostly reflect the model's own mistakes, so removing them does not help. The useful signal is the overlap with the VADER disagreements: if those are flagged more often than other tweets, the two independent checks point at the same questionable labels. **Re-run this section once the Sentence-BERT vectors from `10_emb_sbert` are available**; it automatically uses the embedding with the best CV score.